# Auto Label
This notebook is used to:
- Automatically generate a caption and a label for each image using a free model
- I'm too broke for any openai API calls :(( so i'll just use Qwen-2.5-VL-3B model

- Then, we can use the generated labels to **train a smaller model or a BERT for better accuracy**
- Manually correct mislabled image afterwards, check **data_cleaning.ipynb** for more info

In [1]:
!source ../miniconda3/bin/activate
!conda --version

conda 24.11.3


In [6]:
import os
import json
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# --------------------------------------------------
# Configuration
# --------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
IMAGE_DIR = "fire_dataset_3"
OUTPUT_CSV = "fire_captions_3.csv"

In [3]:
# --------------------------------------------------
# Load processor and model
# --------------------------------------------------
# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("Warning: Running on CPU may be slow. GPU is recommended.")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",                 # 🧠 This spreads across GPUs
    torch_dtype=torch.bfloat16,         # Use bfloat16 for lower memory
)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [7]:
import re, json
# Directory of images
records = []

# Fire analysis prompt
system_prompt = (
    "You are a visual analyst evaluating an image for signs of fire and the surrounding context. "
    "Do the following tasks:\n"
    "1: Summarize what you see in the image. Describe the environment, key objects, people, and any signs of fire or smoke.\n"
    "2: Based on your summary, classify the fire situation: "
    "no fire(e.g., fire alarm, fire distinguisher,..), controlled fire (e.g., fireplace emitting, campfire, cooking, candles, match stick, lighter..) or a dangerous/uncontrolled fire (e.g., curtains on fire, smoke on ceiling, couch on fire, bed sheet on fire, spreading fire on furniture..)?\n"
    "Return only this JSON format:\n"
    "{ \"caption\": \"...\", \"label\": \"no fire\"|\"controlled fire\"|\"dangerous fire\" }"
)

# Loop through image files
for fname in os.listdir(IMAGE_DIR):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    image_path = os.path.join(IMAGE_DIR, fname)
    try:
        image = Image.open(image_path).convert("RGB")

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": system_prompt}
            ]
        }]

        # Generate prompt and inputs
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(device)

        # Run model
        generate_ids = model.generate(**inputs, max_new_tokens=1024)
        output = processor.tokenizer.batch_decode(generate_ids, skip_special_tokens=True)[0]
        if output.startswith("system"):
            output = output.split("assistant", 1)[-1].strip()
        # Clean up any system/user/assistant roles
        clean_output = re.sub(r'^(system|user|assistant)\n', '', output, flags=re.MULTILINE).strip()

        # Match the last complete JSON block
        matches = re.findall(r'\{[^{}]+\}', clean_output, re.DOTALL)
        if matches:
            try:
                result = json.loads(matches[-1])  # take the last match (most likely correct)
                caption = result.get("caption", "").strip()
                label = result.get("label", "").strip()
            except Exception as e:
                print(f"❌ Failed to parse JSON for {fname}: {e}")
                caption = clean_output
                label = "unknown"
        else:
            print(f"❌ No JSON block found in output for {fname}")
            caption = clean_output
            label = "unknown"

        print(f"✅ {fname} - {label} - {caption} ")
        records.append({
            "image_path": image_path,
            "caption": caption,
            "label": label
        })

    except Exception as e:
        print(f"❌ Error processing {fname}: {e}")

✅ image1192.jpg - controlled fire - The image shows two gas burners with blue flames burning on a stove. The flames are bright and intense, indicating active combustion. 
✅ image1165.jpg - controlled fire - A hand holding a black lighter with a lit flame against a plain background. 
✅ image1200.jpg - controlled fire - A beach scene with a small bonfire burning on the sand, surrounded by a blanket. Two marshmallows are being roasted over the flames, with a large rock formation visible in the background under a clear sky. 
✅ image1175.jpg - no fire - The image shows a well-decorated dining table with elegant candle holders, wine glasses, and place settings. The room has a cozy ambiance with soft lighting and a comfortable seating area in the background. 
✅ image1207.jpg - controlled fire - The image shows several lit candles with flames burning brightly against a dark background. 
✅ image1163.jpg - controlled fire - A hand holding a lit lighter with a flame visible, set against a dark ba

In [8]:
df = pd.DataFrame(records)[['image_path', 'label', 'caption']]  # Specify column order
df.to_csv(OUTPUT_CSV, index=False)
print(f"📁 Saved results to {OUTPUT_CSV}")

📁 Saved results to fire_captions_3.csv
